(external-model-to-mm)
# Push external model events to MLRun model monitoring

This section illustrates the two ways an external model can push prediction
events into MLRun's model-monitoring stream pod over HTTP:

- **Env-driven flow** — `fn.setup_model_monitoring()` registers a `USER_EP`
   model endpoint and injects `MODEL_MONITORING_URL`, `MODEL_ENDPOINT_UID` and
   `MODEL_ENDPOINT_NAME` env vars into the Nuclio pod at deploy time. The
   handler reads them from `os.environ`.

- **Event-driven flow** — you register the endpoint and fetch the stream pod
   URL with `project.create_user_model_endpoint(...)` and
   `project.get_model_monitoring_url()`, then pass them inside the invocation
   body. This paradigm does not use env vars, or a `setup_model_monitoring()` call.

The same `user_application.py` handler supports both modes — values from the
event body win, env vars are the fallback.

After both deploys we verify that the events landed in the TSDB and that the
default `histogram-data-drift` monitoring app produced results for each
endpoint.

**In this section**

- [SDK](#sdk)
- [Project setup](#project-setup)
- [Env-driven flow](#env-driven-flow)
- [Event-driven flow](#event-driven-flow)
- [](    

## SDK
- {py:meth}`~mlrun.projects.MlrunProject.create_user_model_endpoint`: create an endpoint
- {py:meth}`~mlrun.projects.MlrunProject.get_model_monitoring_url`: retrieve the URL of the serving pod’s HTTP trigger

## Project setup

Create the project and select the MLRun image.

In [ ]:
%config Completer.use_jedi = False

import json
import time
from datetime import UTC, datetime, timedelta

import mlrun
import mlrun.model_monitoring
from mlrun import get_or_create_project
from mlrun.common.schemas.model_monitoring import ModelEndpointInstruction
from mlrun.datastore.datastore_profile import DatastoreProfileV3io

image = "mlrun/mlrun"
project_name = "streamline-demo"
project = get_or_create_project(project_name, context="./", allow_cross_project=True)
project_name

Configure model monitoring credentials and enable monitoring

Register two V3IO datastore profiles (one for the stream, one for the TSDB)
and call `enable_model_monitoring`. The default `histogram-data-drift`
application is deployed automatically.

In [ ]:
tsdb_profile = DatastoreProfileV3io(
    name="v3io-tsdb-profile",
    v3io_access_key=mlrun.mlconf.get_v3io_access_key(),
)
stream_profile = DatastoreProfileV3io(
    name="v3io-stream-profile",
    v3io_access_key=mlrun.mlconf.get_v3io_access_key(),
)

project.register_datastore_profile(tsdb_profile)
project.register_datastore_profile(stream_profile)

project.set_model_monitoring_credentials(
    stream_profile_name=stream_profile.name,
    tsdb_profile_name=tsdb_profile.name,
    replace_creds=True,
)

project.enable_model_monitoring(base_period=1, image=image, wait_for_deployment=True)

## Build a small synthetic input batch

`user_application.py` is a toy credit-approval model with feature schema
`[age, income, credit_score, balance]` and label schema `[approved]`.

Each row produces one prediction event that gets POSTed to the stream pod.

In [ ]:
feature_names = ["age", "income", "credit_score", "balance"]
label_names = ["approved"]

# 20 rows — large enough for the histogram drift app to produce a result.
data_points = [
    [float(30 + i), float(50_000 + 1_000 * i), float(600 + 5 * i), float(2_000 + 100 * i)]
    for i in range(20)
]
len(data_points)

## Env-driven flow

`fn.setup_model_monitoring(...)` tells MLRun to register a `USER_EP` model
endpoint when the function deploys and to inject the routing env vars into
the Nuclio pod. The handler then reads them from `os.environ`.

In [ ]:
code_path = ruser_application.py"
endpoint_name_1 = "http-ingest-ep-env"

func1 = project.set_function(
    func=code_path,
    name="user-application",
    image=image,
    kind="remote",
)
func1.setup_model_monitoring(
    general_model_endpoint_instructions=ModelEndpointInstruction(
        name=endpoint_name_1,
        input_schema=feature_names,
        output_schema=label_names,
    ),
)
func1.deploy()

### Invoke deploy #1

The body only carries `inputs` — the routing comes from the injected env vars.

In [ ]:
time.sleep(5)  # give the endpoint registration a moment to settle

run_db = mlrun.get_run_db()
endpoints = run_db.list_model_endpoints(
    project=project_name, names=endpoint_name_1
).endpoints
assert endpoints, f"No endpoint with name {endpoint_name_1!r} was registered"
endpoint_uid_1 = endpoints[0].metadata.uid
print(f"deploy #1 endpoint: name={endpoint_name_1!r} uid={endpoint_uid_1!r}")

result_1 = func1.invoke("/", body=json.dumps({"inputs": data_points}))
result_1

## Event-driven flow
In this flow you configure model emdpoint, the name and UID, and then resuest the model monitoring URL. 

the use `create_user_model_endpoint` and `get_model_monitoring_url` to 

This time we drive the wiring ourselves:

* `project.create_user_model_endpoint(...)` registers the `USER_EP` and returns `(name, uid)`.
* `project.get_model_monitoring_url()` returns the internal HTTP URL of the monitoring stream pod.
* Deploy the **same** `user_application.py` without calling `setup_model_monitoring()`. No env vars are injected.
* On invoke, pass `model_endpoint_uid`, `model_endpoint_name` and `model_monitoring_url` inside the request body. The handler resolves the
  routing from the body.

In [ ]:
from mlrun.common.schemas.model_monitoring.constants import ModelEndpointCreationStrategy
func2 = project.set_function(
    func=code_path,
    name="user-application-2",
    image=image,
    kind="remote",
)
func2.save()

endpoint_name_2, endpoint_uid_2 = project.create_user_model_endpoint(
    name="http-ingest-ep-event",
    input_schema=feature_names,
    output_schema=label_names,
    function_name="user-application-2",
    function_tag="latest",
    creation_strategy=ModelEndpointCreationStrategy.OVERWRITE,
)
monitoring_url = project.get_model_monitoring_url()
print(f"deploy #2 endpoint: name={endpoint_name_2!r} uid={endpoint_uid_2!r}")
print(f"monitoring URL: {monitoring_url}")
assert monitoring_url, "project.get_model_monitoring_url() returned no URL"

# NB: no func2.setup_model_monitoring() — the second deploy is event-driven.
func2.deploy()

### Invoke deploy #2

Routing is passed inside the request body, so the same handler picks the
endpoint and stream URL from the event.

In [ ]:
result_2 = func2.invoke(
    "/",
    body=json.dumps(
        {
            "inputs": data_points,
            "model_endpoint_uid": endpoint_uid_2,
            "model_endpoint_name": endpoint_name_2,
            "model_monitoring_url": monitoring_url,
        }
    ),
)
result_2

## 7. Deploy #3 — ApplicationRuntime

Same flow, this time with **`kind="application"`** instead of `kind="remote"`.
The user's HTTP server (a tiny Flask app, `user_application_flask.py`) runs
inside the **sidecar container** of the application pod; a reverse-proxy
container in front of it routes requests through a Nuclio HTTP trigger.

`deploy(track_models=True)` triggers `super().deploy()` (`RemoteRuntime`),
which calls `setup_model_monitoring()` if no instructions are set and then
posts to the API server. The server registers the `USER_EP` and injects
`MODEL_MONITORING_URL` / `MODEL_ENDPOINT_UID` / `MODEL_ENDPOINT_NAME` into
the function spec — and Nuclio propagates these to **both** the reverse-proxy
*and* the sidecar containers, so the Flask app reads them straight from
`os.environ`. (No mlrun needed inside the sidecar — we pass `with_mlrun=False`
to keep the image lean.)

**No source-loader init container.** Instead of `with_source_archive(...)`
(which would require an `mlrun load-source` init container on the
cluster's datanode-registry), we bake the Flask app into the sidecar image
at build time via `spec.build.commands`. First deploy takes a few minutes
to build the image with `Flask + requests`; redeploys are fast.

In [ ]:
import base64
import pathlib

endpoint_name_3 = "http-ingest-ep-application-1"

# Bake the Flask app into the sidecar image at /app/user_application_flask.py
# via a build command. Avoids ApplicationRuntime's source-loader init container,
# whose default image (mlconf.default_base_image at the SDK version tag) is not
# present in this cluster's datanode-registry mirror.
flask_b64 = base64.b64encode(
    pathlib.Path("user_application_flask.py").read_bytes()
).decode()

func3 = project.set_function(
    name="user-application-app-1",
    kind="application",
    image=image,
    requirements=["Flask==3.0.0", "requests"],
)
func3.set_internal_application_port(5000)
func3.spec.build.commands = [
    "mkdir -p /app",
    f"echo {flask_b64} | base64 -d > /app/user_application_flask.py",
]
func3.spec.command = "/bin/sh"
func3.spec.args = [
    "-c",
    "cd /app && python -m flask --app=user_application_flask run "
    "--host=0.0.0.0 --port=5000",
]
func3.setup_model_monitoring(
    general_model_endpoint_instructions=ModelEndpointInstruction(
        name=endpoint_name_3,
        input_schema=feature_names,
        output_schema=label_names,
    ),
)
func3.deploy(with_mlrun=False)

### Fastest path: deploy an application with monitoring via `track_models=True`

When you don't need a custom endpoint name, schemas, or feature stats, you can skip
`setup_model_monitoring()` entirely and just pass `track_models=True` to `deploy()`.
The runtime fills in the gaps for you.

```python
import base64
import pathlib

# Bake the Flask app into the sidecar image at /app/user_application_flask.py
# via a build command. Avoids ApplicationRuntime's source-loader init container,
# whose default image (mlconf.default_base_image at the SDK version tag) is not
# present in this cluster's datanode-registry mirror.
flask_b64 = base64.b64encode(
    pathlib.Path("user_application_flask.py").read_bytes()
).decode()

func3 = project.set_function(
    name="user-application-app-1",
    kind="application",
    image=image,
    requirements=["Flask==3.0.0", "requests"],
)
func3.set_internal_application_port(5000)
func3.spec.build.commands = [
    "mkdir -p /app",
    f"echo {flask_b64} | base64 -d > /app/user_application_flask.py",
]
func3.spec.command = "/bin/sh"
func3.spec.args = [
    "-c",
    "cd /app && python -m flask --app=user_application_flask run "
    "--host=0.0.0.0 --port=5000",
]
func3.deploy(track_models=True, with_mlrun=False)
```

#### What happens behind the scenes

`func3.deploy(track_models=True, ...)` triggers this chain (see
`mlrun/runtimes/nuclio/function.py:969`):

1. **Auto-`setup_model_monitoring()`** — because `spec.model_endpoints_instructions`
   is empty, `deploy()` calls `setup_model_monitoring()` with **no arguments**.
   That creates one **default model endpoint** for this function:
   - auto-generated endpoint name and UID
   - no `input_schema` / `output_schema` (you lose drift on individual features,
     but ingestion still works)
   - `spec.track_models` is flipped to `True`
2. **Server creates the endpoint at deploy time** — the API service registers the
   model endpoint row in the DB and returns its UID in the deploy background task.
3. **MM env vars get injected into the function pod** — the server adds
   `MODEL_MONITORING_URL`, `MODEL_ENDPOINT_UID`, and `MODEL_ENDPOINT_NAME` to
   `spec.env`. Nuclio's processor then renders the pod template so those vars
   appear on **every container in the pod**, including the application sidecar
   running Flask (verified by dumping the pod spec).
4. **Your Flask code reads the env vars** — inside the sidecar,
   `user_application_flask.py` reads `MODEL_MONITORING_URL` + `MODEL_ENDPOINT_UID`
   and POSTs prediction events to the model-monitoring stream pod's HTTP trigger.
   No code path through `mlrun.serving` is involved (and `with_mlrun=False` keeps
   mlrun out of the sidecar image entirely).
5. **Stream pod → TSDB** — the stream pod ingests the HTTP events, writes them to
   the monitoring stream, and the monitoring application (e.g.
   `NoCheckDemoMonitoringApp`) consumes them and writes results to TSDB.

#### When to use this vs. `setup_model_monitoring(...)`

- **Use `track_models=True` alone** when you just want monitoring "on" with sensible
  defaults — quickest demo path, one knob.
- **Call `setup_model_monitoring(general_model_endpoint_instructions=...)`** when
  you need a stable/custom endpoint name, explicit input/output schemas (for
  per-feature drift), or to register multiple endpoints on the same function.


In [ ]:
from mlrun import get_model_monitoring_url

get_model_monitoring_url("streamline-demo")

### Invoke deploy #3

Same env-driven shape as deploy #1 — the Flask app reads the routing values
from `os.environ` (injected by `track_models=True` and propagated by Nuclio
to the sidecar). The request body only carries `inputs`.

In [ ]:
time.sleep(5)

endpoints = run_db.list_model_endpoints(
    project=project_name, names=endpoint_name_3
).endpoints
assert endpoints, f"No endpoint with name {endpoint_name_3!r} was registered"
endpoint_uid_3 = endpoints[1].metadata.uid
print(f"deploy #3 endpoint: name={endpoint_name_3!r} uid={endpoint_uid_3!r}")

result_3 = func3.invoke("/", body={"inputs": data_points}, verify=False)
result_3

## 8. Verify prediction events landed in the TSDB

The stream pod writes prediction events to the TSDB in batches. We poll
`get_last_request` and `read_predictions` for both endpoints. The wait time
matches what the system tests use: the parquet batching + writer-graph flush
windows configured on the cluster.

In [ ]:
import mlrun.common.schemas.model_monitoring as mm_schemas

tsdb = mlrun.model_monitoring.get_tsdb_connector(
    project=project_name, profile=tsdb_profile
)

predictions_wait = (
    mlrun.mlconf.model_endpoint_monitoring.parquet_batching_timeout_secs
    + mlrun.mlconf.model_endpoint_monitoring.writer_graph.flush_after_seconds
)
print(f"Waiting up to ~{predictions_wait}s for predictions to flush to TSDB...")
# time.sleep(predictions_wait)

# The high-level `tsdb.read_predictions(...)` helper passes datetime objects to
# v3io_frames, which the build on this cluster doesn't accept. Go straight to
# the frames client with relative-time strings ("now-1h", "now") which is the
# canonical v3io_frames format.
preds_table = tsdb.tables[mm_schemas.V3IOTSDBTables.PREDICTIONS]


def read_predictions_direct(uid: str):
    return tsdb.frames_client.read(
        backend="tsdb",
        table=preds_table,
        start="now-172h",
        end="now",
        filter=f"endpoint_id=='{uid}'",
    )


for label, uid in [
    ("deploy #1 (env)", endpoint_uid_1),
    ("deploy #2 (event)", endpoint_uid_2),
    ("deploy #3 (application)", endpoint_uid_3),
]:
    lr = tsdb.get_last_request(endpoint_ids=uid)
    assert lr and uid in lr, f"No last_request for {uid} yet"
    pred = read_predictions_direct(uid)
    print(f"{label}: last_request={lr[uid]}")
    print(f"{label}: predictions rows = {len(pred)}")
    assert not pred.empty, f"No predictions in TSDB for {uid} yet"
    print(pred.head())
    print("-" * 60)

## 9. Verify the `monitoring-test` app produced results

The demo `monitoring-test` application runs on the schedule defined by
`base_period` (1 minute here) and emits two result rows per cycle. We query
the TSDB `app_results` table directly via the frames client (same workaround
as the predictions check) and confirm rows for both endpoints.

In [ ]:
app_wait = 2 * 60  # base_period=1 minute, give the app two cycles
print(f"Waiting up to ~{app_wait}s for monitoring-test to produce results...")
# time.sleep(app_wait)

# Same workaround as for predictions — go direct to the frames client with
# relative-time strings instead of `tsdb.get_results_metadata(...)`.
app_table = tsdb.tables[mm_schemas.V3IOTSDBTables.APP_RESULTS]


def read_app_results_direct(uid: str):
    return tsdb.frames_client.read(
        backend="tsdb",
        table=app_table,
        start="now-120h",
        end="now",
        columns=[mm_schemas.ResultData.RESULT_KIND],
        filter=f"endpoint_id=='{uid}'",
        aggregators="last",
    )


for label, uid in [
    ("deploy #1 (env)", endpoint_uid_1),
    ("deploy #2 (event)", endpoint_uid_2),
    ("deploy #3 (application)", endpoint_uid_3),
]:
    df = read_app_results_direct(uid)
    print(f"{label}: app-result rows = {len(df)}")
    if not df.empty:
        print(df.head())
    print("-" * 60)
    assert not df.empty, f"No app results in TSDB for {uid}"
    # The frames index for app_results is (endpoint_id, application_name, result_name, time).
    app_names = df["application_name"].values
    assert "monitoring-test" in app_names, (
        f"monitoring-test app did not produce results for {uid} "
        f"(saw: {sorted(set(app_names))})"
    )

In [ ]:
df

In [ ]:
df["application_name"].values